In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/indian-railways-predict-train-delay/ir_test.csv
/kaggle/input/competitions/indian-railways-predict-train-delay/ir_train.csv
/kaggle/input/competitions/indian-railways-predict-train-delay/ir_sample_submission.csv
/kaggle/input/competitions/indian-railways-predict-train-delay/ir_data_dictionary.csv


In [ ]:
# !pip install -q "xgboost>=2.0" "scikit-learn>=1.4" pandas pyarrow joblib

In [86]:
from pathlib import Path
import gc
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

In [87]:
DATA_DIR = Path("/kaggle/input/competitions/indian-railways-predict-train-delay")

TRAIN_PATH = DATA_DIR / "ir_train.csv"
TEST_PATH = DATA_DIR / "ir_test.csv"

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False, parse_dates=["departure_date"])

test = pd.read_csv(TEST_PATH, low_memory=False, parse_dates=["departure_date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())

Train shape: (1500000, 45)
Test shape: (375000, 42)


,journey_id,train_number,train_type,departure_date,year,month,day_of_week,departure_hour,is_weekend,is_night_departure,...,is_rake_shared,maintenance_score,seat_utilisation_pct,is_overloaded,late_incoming_rake,is_special_train,route_historical_ontime_pct,primary_delay_cause,delay_minutes,is_delayed
0,IR01717809,12536,Superfast Express,2023-09-18,2023,9,0,7,0,0,...,1,5.5,88.8,0,0,0,67.8,Flooding / Waterlogging,110,1
1,IR00931776,12806,Superfast Express,2020-04-09,2020,4,3,8,0,0,...,1,6.6,83.6,0,0,0,57.0,On Time,3,0
2,IR00096904,22416,Vande Bharat Express,2024-12-13,2024,12,4,18,0,0,...,0,4.1,81.6,0,0,0,81.8,On Time,5,0
3,IR00047736,12250,Garib Rath Express,2018-11-29,2018,11,3,18,0,0,...,0,5.5,82.1,0,1,0,52.3,Track Congestion,135,1
4,IR00726037,74588,DEMU/MEMU,2020-06-18,2020,6,3,5,0,0,...,1,8.0,58.3,0,1,0,64.3,Track Congestion,154,1


In [89]:
print("Train-only columns:")
print(sorted(set(train.columns) - set(test.columns)))

print("\nTest-only columns:")
print(sorted(set(test.columns) - set(train.columns)))

Train-only columns:
['delay_minutes', 'is_delayed', 'primary_delay_cause']

Test-only columns:
[]


In [ ]:
TARGET = "is_delayed"

LEAKAGE_COLUMNS = [
    "primary_delay_cause",
    "delay_minutes",
]

ID_COLUMNS = [
    "journey_id",
]

FEATURE_COLUMNS = [col for col in test.columns if col not in ID_COLUMNS]

In [91]:
print("Number of model features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Number of model features: 41
['train_number', 'train_type', 'departure_date', 'year', 'month', 'day_of_week', 'departure_hour', 'is_weekend', 'is_night_departure', 'is_peak_hour', 'is_festival_season', 'season', 'zone', 'zone_abbr', 'source_station_category', 'destination_station_category', 'distance_km', 'num_scheduled_stops', 'scheduled_travel_hours', 'track_doubled', 'is_hdn_route', 'traction_type', 'is_electrified', 'psr_count', 'is_circular_route', 'is_monsoon_season', 'is_fog_risk', 'fog_risk_score', 'zone_fog_index', 'zone_congestion_index', 'season_severity_score', 'loco_age_years', 'coach_age_years', 'has_lhb_coaches', 'is_rake_shared', 'maintenance_score', 'seat_utilisation_pct', 'is_overloaded', 'late_incoming_rake', 'is_special_train', 'route_historical_ontime_pct']


## Feature engineering


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "departure_date" in df.columns:
        date = pd.to_datetime(df["departure_date"], errors="coerce")

        df["departure_day_of_year"] = date.dt.dayofyear.astype("float32")
        df["departure_week_of_year"] = date.dt.isocalendar().week.astype("float32")
        df["departure_quarter"] = date.dt.quarter.astype("float32")

        reference_date = pd.Timestamp("2015-01-01")
        df["departure_days_from_reference"] = (date - reference_date).dt.days.astype(
            "float32"
        )

        df = df.drop(columns=["departure_date"])

    return df

In [93]:
train = add_features(train)
test = add_features(test)

In [ ]:
FEATURE_COLUMNS = [col for col in test.columns if col != "journey_id"]

print("Final feature count:", len(FEATURE_COLUMNS))

Final feature count: 44


In [95]:
train = train.sort_values("departure_days_from_reference").reset_index(drop=True)

split_index = int(len(train) * 0.80)

train_df = train.iloc[:split_index].copy()
valid_df = train.iloc[split_index:].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(valid_df))

Train rows: 1200000
Validation rows: 300000


In [96]:
X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET].astype("int8")

X_valid = valid_df[FEATURE_COLUMNS].copy()
y_valid = valid_df[TARGET].astype("int8")

X_test = test[FEATURE_COLUMNS].copy()

print("Positive class rate in train:", y_train.mean())
print("Positive class rate in validation:", y_valid.mean())

Positive class rate in train: 0.7183475
Positive class rate in validation: 0.72104


In [ ]:
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

for col in categorical_columns:
    combined_categories = (
        pd.concat([X_train[col], X_valid[col], X_test[col]], ignore_index=True)
        .astype("string")
        .fillna("__MISSING__")
        .unique()
    )

    category_dtype = pd.CategoricalDtype(categories=combined_categories)

    X_train[col] = (
        X_train[col].astype("string").fillna("__MISSING__").astype(category_dtype)
    )

    X_valid[col] = (
        X_valid[col].astype("string").fillna("__MISSING__").astype(category_dtype)
    )

    X_test[col] = (
        X_test[col].astype("string").fillna("__MISSING__").astype(category_dtype)
    )

Categorical columns:
['train_type', 'season', 'zone', 'zone_abbr', 'source_station_category', 'destination_station_category', 'traction_type']


In [98]:
positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

scale_pos_weight = negative_count / positive_count

print("Positive samples:", positive_count)
print("Negative samples:", negative_count)
print("scale_pos_weight:", scale_pos_weight)

Positive samples: 862017
Negative samples: 337983
scale_pos_weight: 0.3920839148183852


## Model creation


In [ ]:
model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=10,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=5.0,
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=100,
)

In [ ]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=100,
)

[0]	validation_0-auc:0.89891
[100]	validation_0-auc:0.91761
[200]	validation_0-auc:0.92179
[300]	validation_0-auc:0.92249
[400]	validation_0-auc:0.92263
[500]	validation_0-auc:0.92260
[508]	validation_0-auc:0.92260


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=100,
              enable_categorical=True, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=10, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=3000,
              n_jobs=-1, num_parallel_tree=None, ...)

## Model Evaluation


In [ ]:
valid_proba = model.predict_proba(X_valid)[:, 1]

roc_auc = roc_auc_score(y_valid, valid_proba)

pr_auc = average_precision_score(y_valid, valid_proba)

print(f"Validation ROC-AUC: {roc_auc:.6f}")
print(f"Validation PR-AUC:  {pr_auc:.6f}")

Validation ROC-AUC: 0.922634
Validation PR-AUC:  0.967890


## Threshold


In [ ]:
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (valid_proba >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print(classification_report(y_valid, pred, digits=4))

BEST_THRESHOLD = 0.50


Threshold: 0.3
              precision    recall  f1-score   support

           0     0.7669    0.7215    0.7435     83688
           1     0.8947    0.9152    0.9048    216312

    accuracy                         0.8611    300000
   macro avg     0.8308    0.8183    0.8241    300000
weighted avg     0.8590    0.8611    0.8598    300000


Threshold: 0.4
              precision    recall  f1-score   support

           0     0.7119    0.7972    0.7521     83688
           1     0.9177    0.8752    0.8960    216312

    accuracy                         0.8534    300000
   macro avg     0.8148    0.8362    0.8240    300000
weighted avg     0.8603    0.8534    0.8558    300000


Threshold: 0.5
              precision    recall  f1-score   support

           0     0.6595    0.8539    0.7442     83688
           1     0.9362    0.8294    0.8796    216312

    accuracy                         0.8362    300000
   macro avg     0.7978    0.8417    0.8119    300000
weighted avg     0.8590   

In [103]:
BEST_THRESHOLD = 0.45

In [ ]:
importance = pd.DataFrame(
    {"feature": model.feature_names_in_, "importance": model.feature_importances_}
).sort_values("importance", ascending=False)

display(importance.head(20))

importance.to_csv(MODEL_DIR / "feature_importance.csv", index=False)

,feature,importance
24,is_monsoon_season,0.195494
37,late_incoming_rake,0.136740
29,season_severity_score,0.124188
10,season,0.122845
3,month,0.071213
39,route_historical_ontime_pct,0.069510
18,track_doubled,0.038608
25,is_fog_risk,0.033714
1,train_type,0.025181
21,is_electrified,0.014876


## Save the model and metadata


In [ ]:
model.save_model(MODEL_DIR / "delay_risk_xgboost.json")

model_metadata = {
    "model_type": "XGBClassifier",
    "model_version": "delay-risk-v1",
    "target": "is_delayed",
    "positive_class_definition": "arrival delayed by more than 15 minutes",
    "threshold": BEST_THRESHOLD,
    "feature_columns": FEATURE_COLUMNS,
    "categorical_columns": categorical_columns,
    "excluded_leakage_columns": LEAKAGE_COLUMNS,
}

with open(MODEL_DIR / "delay_risk_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

## Test model on the provided test dataset


In [ ]:
# Test model on the provided test dataset
test_delay_probability = model.predict_proba(X_test)[:, 1]

test_delay_prediction = (test_delay_probability >= BEST_THRESHOLD).astype(int)

# Create results:

test_results = pd.DataFrame(
    {
        "journey_id": test["journey_id"],
        "delay_probability": test_delay_probability,
        "predicted_is_delayed": test_delay_prediction,
    }
)

display(test_results.head())

# Save:

test_results.to_csv(MODEL_DIR / "test_delay_predictions.csv", index=False)

,journey_id,delay_probability,predicted_is_delayed
0,IR01679028,0.091679,0
1,IR01531216,0.123388,0
2,IR01076165,0.989603,1
3,IR00956901,0.258204,0
4,IR00470951,0.890089,1


In [ ]:
# Sanity-check the model
print(test_results[["delay_probability", "predicted_is_delayed"]].describe())

print(
    "Predicted delayed percentage:", test_results["predicted_is_delayed"].mean() * 100
)

       delay_probability  predicted_is_delayed
count      375000.000000         375000.000000
mean            0.623532              0.660296
std             0.353535              0.473609
min             0.001171              0.000000
25%             0.272997              0.000000
50%             0.751991              1.000000
75%             0.958785              1.000000
max             0.999677              1.000000
Predicted delayed percentage: 66.0296


## Load the saved model and test inference


In [ ]:
from xgboost import XGBClassifier

loaded_model = XGBClassifier()
loaded_model.load_model(MODEL_DIR / "delay_risk_xgboost.json")

# Test one batch:

sample_prediction = loaded_model.predict_proba(X_test.head(10))[:, 1]

print(sample_prediction)

[0.09167944 0.12338774 0.9896026  0.2582045  0.8900891  0.98721385
 0.83564925 0.9160019  0.8220736  0.79079705]


## This model is not the final ETAPredictionResult model.

### It predicts: P(train will be >15 minutes delayed)


Key conclusion: this first model should be treated as a historical delay-risk model trained from the available dataset. In the next notebook, we should build the schema-aligned adapter and ETA prediction model/pipeline, so that the final system actually accepts ETAPredictionRequest and returns ETAPredictionResult. This prevents incorrectly pretending that the Kaggle-style dataset directly trains a model for live station-level ETA prediction.


# Next notebook: Schema-Aligned ETA Pipeline

### This notebook creates the actual inference layer aligned with your ETAPredictionRequest and ETAPredictionResult schemas.


- /kaggle/input/competitions/indian-railways-predict-train-delay/ir_test.csv
- /kaggle/input/competitions/indian-railways-predict-train-delay/ir_train.csv
- /kaggle/input/competitions/indian-railways-predict-train-delay/ir_sample_submission.csv
- /kaggle/input/competitions/indian-railways-predict-train-delay/ir_data_dictionary.csv


In [ ]:
"""ETAPredictionRequest
│
├── Scheduled ETA component
│
├── Current delay propagation
│
├── Live speed adjustment
│
├── Historical section adjustment
│
├── Weather adjustment
│
├── Congestion adjustment
│
└── XGBoost historical delay risk
            │
            ▼
  predicted_remaining_minutes
            │
            ▼
   ETAPredictionResult"""

# must have
"""
models/
├── delay_risk_xgboost.json
└── delay_risk_metadata.json"""

'\nmodels/\n├── delay_risk_xgboost.json\n└── delay_risk_metadata.json'

In [ ]:
# 1. Imports
from pathlib import Path
from datetime import datetime, timezone
import json
import math

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

# 2. Load the historical XGBoost delay-risk model
MODEL_DIR = Path("./models")

RISK_MODEL_PATH = MODEL_DIR / "delay_risk_xgboost.json"
METADATA_PATH = MODEL_DIR / "delay_risk_metadata.json"

with open(METADATA_PATH, "r") as f:
    risk_metadata = json.load(f)

risk_model = XGBClassifier(enable_categorical=True)

risk_model.load_model(RISK_MODEL_PATH)

print("Loaded model:", risk_metadata["model_version"])


# 3. Load the historical dataset

# The dataset is needed here only to create historical context.

DATA_DIR = Path("/kaggle/input/competitions/indian-railways-predict-train-delay")


TRAIN_PATH = DATA_DIR / "ir_train.csv"

historical_data = pd.read_csv(TRAIN_PATH, low_memory=False)

print(historical_data.shape)

Loaded model: delay-risk-v1
(1500000, 45)


### 4. Create historical lookup tables

- The deployed API receives: train_id

- while the historical dataset uses: train_number

- For now, we assume: train_id == train_number


In [ ]:
historical_data["train_number"] = historical_data["train_number"].astype(str)

In [ ]:
# 5. Train-level historical statistics
train_history = (
    historical_data.groupby("train_number")
    .agg(
        historical_delay_rate=("is_delayed", "mean"),
        historical_avg_delay_minutes=("delay_minutes", "mean"),
        historical_avg_speed_kmph=("distance_km", lambda x: np.nan),
    )
    .reset_index()
)

# The average speed calculation should use total distance divided by travel hours.

train_history = (
    historical_data.groupby("train_number")
    .agg(
        historical_delay_rate=("is_delayed", "mean"),
        historical_avg_delay_minutes=("delay_minutes", "mean"),
        historical_avg_distance_km=("distance_km", "mean"),
        historical_avg_travel_hours=("scheduled_travel_hours", "mean"),
        historical_ontime_pct=("route_historical_ontime_pct", "mean"),
    )
    .reset_index()
)

# Now calculate historical speed:

train_history["historical_avg_speed_kmph"] = (
    train_history["historical_avg_distance_km"]
    / train_history["historical_avg_travel_hours"]
)

train_history["historical_avg_speed_kmph"] = train_history[
    "historical_avg_speed_kmph"
].replace([np.inf, -np.inf], np.nan)

# 6. Global fallback statistics


In [ ]:
# 6. Global fallback statistics

# A train may not exist in the historical data.

GLOBAL_HISTORICAL_DELAY_RATE = historical_data["is_delayed"].mean()

GLOBAL_HISTORICAL_DELAY_MINUTES = historical_data["delay_minutes"].mean()

GLOBAL_HISTORICAL_SPEED_KMPH = (
    (historical_data["distance_km"] / historical_data["scheduled_travel_hours"])
    .replace([np.inf, -np.inf], np.nan)
    .median()
)

GLOBAL_HISTORICAL_ONTIME_PCT = historical_data["route_historical_ontime_pct"].mean()

## SAVE REQUIRED BACKEND ARTIFACTS


In [ ]:
# SAVE REQUIRED BACKEND ARTIFACTS

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)


# 1. Save train history lookup

train_history.to_parquet(MODEL_DIR / "train_history_lookup.parquet", index=False)

print("Saved:", MODEL_DIR / "train_history_lookup.parquet")

# 2. Create global defaults dictionary

GLOBAL_DEFAULTS = {
    "historical_delay_rate": float(GLOBAL_HISTORICAL_DELAY_RATE),
    "historical_avg_delay_minutes": float(GLOBAL_HISTORICAL_DELAY_MINUTES),
    "historical_avg_speed_kmph": float(GLOBAL_HISTORICAL_SPEED_KMPH),
    "historical_ontime_pct": float(GLOBAL_HISTORICAL_ONTIME_PCT),
}


# 3. Save global defaults

with open(MODEL_DIR / "global_defaults.json", "w") as f:
    json.dump(GLOBAL_DEFAULTS, f, indent=2)

print("Saved:", MODEL_DIR / "global_defaults.json")


# 4. Verify files

print("\nMODEL DIRECTORY:")

for file in MODEL_DIR.iterdir():
    print(file.name, f"({file.stat().st_size / 1024:.2f} KB)")

Saved: models/train_history_lookup.parquet
Saved: models/global_defaults.json

MODEL DIRECTORY:
train_history_lookup.parquet (363.92 KB)
eta_pipeline_config.json (0.77 KB)
test_delay_predictions.csv (8643.18 KB)
delay_risk_metadata.json (1.47 KB)
global_defaults.json (0.18 KB)
feature_importance.csv (1.24 KB)
delay_risk_xgboost.json (12382.67 KB)


In [ ]:
def get_train_history(train_id: str):
    train_id = str(train_id)

    match = train_history[train_history["train_number"] == train_id]

    if len(match) == 0:
        return {
            "historical_delay_rate": float(GLOBAL_HISTORICAL_DELAY_RATE),
            "historical_avg_delay_minutes": float(GLOBAL_HISTORICAL_DELAY_MINUTES),
            "historical_avg_speed_kmph": float(GLOBAL_HISTORICAL_SPEED_KMPH),
            "historical_ontime_pct": float(GLOBAL_HISTORICAL_ONTIME_PCT),
        }

    row = match.iloc[0]

    return {
        "historical_delay_rate": float(row["historical_delay_rate"]),
        "historical_avg_delay_minutes": float(row["historical_avg_delay_minutes"]),
        "historical_avg_speed_kmph": float(row["historical_avg_speed_kmph"]),
        "historical_ontime_pct": float(row["historical_ontime_pct"]),
    }

### 8. Schema-aligned input example

This exactly follows your ETAPredictionRequest.


In [ ]:
sample_request = {
    "train_id": "12345",
    "target_station_id": "GUWAHATI",
    "observed_at": "2026-09-04T12:30:00+05:30",
    "current_speed_kmph": 58.0,
    "average_speed_last_5_minutes": 55.0,
    "current_delay_minutes": 12.0,
    "distance_to_target_km": 120.0,
    "scheduled_time_to_target_minutes": 130.0,
    "rainfall_mm": 4.0,
    "congestion_score": 0.40,
    "historical_section_average_minutes": 135.0,
}

## 9. Validate required schema fields


In [ ]:
# 9. Validate required schema fields

# For the first prototype, use a lightweight Python validator.

REQUIRED_FIELDS = [
    "train_id",
    "target_station_id",
    "observed_at",
    "current_speed_kmph",
    "average_speed_last_5_minutes",
    "current_delay_minutes",
    "distance_to_target_km",
    "scheduled_time_to_target_minutes",
]


def validate_request(request: dict):
    for field in REQUIRED_FIELDS:

        if field not in request:
            raise ValueError(f"Missing required field: {field}")

    numeric_non_negative_fields = [
        "current_speed_kmph",
        "average_speed_last_5_minutes",
        "distance_to_target_km",
        "scheduled_time_to_target_minutes",
    ]

    for field in numeric_non_negative_fields:

        if request[field] < 0:
            raise ValueError(f"{field} cannot be negative")

    if request.get("rainfall_mm") is not None:

        if request["rainfall_mm"] < 0:
            raise ValueError("rainfall_mm cannot be negative")

    if request.get("congestion_score") is not None:

        if not 0 <= request["congestion_score"] <= 1:

            raise ValueError("congestion_score must be between 0 and 1")

    datetime.fromisoformat(request["observed_at"])

    return True


# Test:

validate_request(sample_request)

print("Request is valid")

Request is valid


### 10. Scheduled baseline

Your request already contains:

scheduled_time_to_target_minutes

This becomes the most important baseline.


In [ ]:
def get_scheduled_baseline(request: dict) -> float:

    return float(request["scheduled_time_to_target_minutes"])

### 11. Current speed estimate

We should combine:

current instantaneous speed
average speed over the last 5 minutes

Recent average speed is generally more stable than instantaneous speed.


In [ ]:
def calculate_live_speed(
    current_speed_kmph: float, average_speed_last_5_minutes: float
) -> float:

    live_speed = 0.40 * current_speed_kmph + 0.60 * average_speed_last_5_minutes

    return max(live_speed, 1.0)

In [ ]:
# 12. Speed-based remaining time


def calculate_speed_eta(distance_to_target_km: float, live_speed_kmph: float) -> float:

    if live_speed_kmph <= 0:
        return None

    hours = distance_to_target_km / live_speed_kmph

    return hours * 60


# Example:

live_speed = calculate_live_speed(
    sample_request["current_speed_kmph"], sample_request["average_speed_last_5_minutes"]
)

speed_eta = calculate_speed_eta(sample_request["distance_to_target_km"], live_speed)

print("Live speed:", live_speed)
print("Speed ETA:", speed_eta)

Live speed: 56.2
Speed ETA: 128.1138790035587


### 13. Historical section component

If your backend provides:

historical_section_average_minutes

use it directly.

Otherwise, fall back to the scheduled time.


In [ ]:
def get_historical_section_eta(request: dict) -> float:

    historical_eta = request.get("historical_section_average_minutes")

    if historical_eta is None:
        return float(request["scheduled_time_to_target_minutes"])

    return float(historical_eta)

### 14. Weather adjustment

This is a simple prototype adjustment.


In [ ]:
def calculate_rainfall_adjustment(rainfall_mm) -> float:

    if rainfall_mm is None:
        return 0.0

    rainfall_mm = float(rainfall_mm)

    if rainfall_mm < 2:
        return 0.0

    if rainfall_mm < 10:
        return 2.0

    if rainfall_mm < 30:
        return 5.0

    return 10.0


# Later, this heuristic should be replaced by a learned weather model
# when real weather and ETA observations are available.

In [ ]:
# 15. Congestion adjustment
def calculate_congestion_adjustment(congestion_score) -> float:

    if congestion_score is None:
        return 0.0

    congestion_score = float(congestion_score)

    return congestion_score * 15.0


# Again, this is a pipeline placeholder, not a learned coefficient.

### 16. Delay-risk estimate

The first notebook trained the XGBoost model on historical features.

However, your live request does not contain all the original model features.

Therefore, we should not invent 40 missing features and pretend the model is making a reliable live prediction.

Instead, we initially use the train-level historical delay statistics:


In [ ]:
def calculate_historical_delay_risk(train_id: str) -> float:

    history = get_train_history(train_id)

    return history["historical_delay_rate"]


# Example:

risk = calculate_historical_delay_risk(sample_request["train_id"])

print("Historical delay risk:", risk)

Historical delay risk: 0.6427255985267035


### 17. Current delay propagation

Current delay is highly relevant to the ETA.

But do not blindly add 100% of the current delay, because trains can recover time.

We propagate part of the delay.


In [ ]:
def calculate_delay_propagation(
    current_delay_minutes: float, delay_risk: float
) -> float:

    if current_delay_minutes <= 0:
        return current_delay_minutes

    propagation_factor = 0.50 + 0.40 * delay_risk

    return current_delay_minutes * propagation_factor


# Examples:

# low risk  → more expected recovery
# high risk → more delay likely to persist

### 18. Combine ETA components

We combine three travel-time estimates:

- scheduled ETA
- historical section ETA
- live speed ETA


In [ ]:
def combine_travel_time_estimates(
    scheduled_eta: float, historical_eta: float, speed_eta: float
) -> float:

    return 0.40 * scheduled_eta + 0.25 * historical_eta + 0.35 * speed_eta

In [ ]:
def calculate_prediction_interval(
    predicted_minutes: float, delay_risk: float, congestion_score
):

    if congestion_score is None:
        congestion_score = 0.0

    uncertainty_factor = 0.08 + 0.12 * delay_risk + 0.08 * congestion_score

    uncertainty = predicted_minutes * uncertainty_factor

    lower_bound = max(0.0, predicted_minutes - uncertainty)

    upper_bound = predicted_minutes + uncertainty

    return (round(lower_bound, 2), round(upper_bound, 2))

## 20. Main schema-aligned ETA pipeline

This is the important function.


In [ ]:
MODEL_VERSION = "hybrid-eta-v1"


def predict_eta(request: dict) -> dict:

    # 1. Validate request
    validate_request(request)

    # 2. Historical context

    history = get_train_history(request["train_id"])

    delay_risk = history["historical_delay_rate"]

    # 3. ETA components

    scheduled_eta = get_scheduled_baseline(request)
    historical_eta = get_historical_section_eta(request)
    live_speed = calculate_live_speed(request["current_speed_kmph"], request["average_speed_last_5_minutes"])
    speed_eta = calculate_speed_eta(request["distance_to_target_km"], live_speed)

    # 4. Base ETA
    base_eta = combine_travel_time_estimates(scheduled_eta, historical_eta, speed_eta)

    # 5. Live adjustments
    delay_adjustment = calculate_delay_propagation(
        request["current_delay_minutes"], delay_risk)

    rainfall_adjustment = calculate_rainfall_adjustment(request.get("rainfall_mm"))

    congestion_adjustment = calculate_congestion_adjustment(
        request.get("congestion_score")
    )

    # 6. Final prediction

    predicted_minutes = (base_eta + delay_adjustment + rainfall_adjustment + congestion_adjustment)

    predicted_minutes = max(0.0, predicted_minutes)

    # 7. Prediction interval

    lower_bound, upper_bound = calculate_prediction_interval(predicted_minutes, delay_risk, request.get("congestion_score"))

    # 8. Result schema

    result = {
        "train_id": request["train_id"],
        "target_station_id": request["target_station_id"],
        "predicted_remaining_minutes": round(predicted_minutes, 2),
        "lower_bound_minutes": lower_bound,
        "upper_bound_minutes": upper_bound,
        "model_version": MODEL_VERSION,
    }

    return result

## 21. Test the full pipeline


In [68]:
sample_request

{'train_id': '12345',
 'target_station_id': 'GUWAHATI',
 'observed_at': '2026-09-04T12:30:00+05:30',
 'current_speed_kmph': 58.0,
 'average_speed_last_5_minutes': 55.0,
 'current_delay_minutes': 12.0,
 'distance_to_target_km': 120.0,
 'scheduled_time_to_target_minutes': 130.0,
 'rainfall_mm': 4.0,
 'congestion_score': 0.4,
 'historical_section_average_minutes': 135.0}

In [ ]:
result = predict_eta(sample_request)

print(json.dumps(result, indent=4))


"""expected result: {
    "train_id": "12345",
    "target_station_id": "GUWAHATI",
    "predicted_remaining_minutes": 142.75,
    "lower_bound_minutes": 122.10,
    "upper_bound_minutes": 163.40,
    "model_version": "hybrid-eta-v1"
}"""

{
    "train_id": "12345",
    "target_station_id": "GUWAHATI",
    "predicted_remaining_minutes": 147.67,
    "lower_bound_minutes": 119.75,
    "upper_bound_minutes": 175.6,
    "model_version": "hybrid-eta-v1"
}


## test on edge cases


In [ ]:
# 22. Test edge cases
# Unknown train
unknown_train_request = sample_request.copy()

unknown_train_request["train_id"] = "UNKNOWN_TRAIN"

unknown_result = predict_eta(unknown_train_request)

print(unknown_result)

NameError: name 'GLOBAL_HISTORICAL_DELAY_RATE' is not defined

In [ ]:
# No rainfall data
no_rain_request = sample_request.copy()

no_rain_request["rainfall_mm"] = None

result_no_rain = predict_eta(no_rain_request)

print(result_no_rain)

{'train_id': '12345', 'target_station_id': 'GUWAHATI', 'predicted_remaining_minutes': 145.67, 'lower_bound_minutes': 118.12, 'upper_bound_minutes': 173.23, 'model_version': 'hybrid-eta-v1'}


In [ ]:
# No congestion data
no_congestion_request = sample_request.copy()

no_congestion_request["congestion_score"] = None

result_no_congestion = predict_eta(no_congestion_request)

print(result_no_congestion)

{'train_id': '12345', 'target_station_id': 'GUWAHATI', 'predicted_remaining_minutes': 141.67, 'lower_bound_minutes': 119.41, 'upper_bound_minutes': 163.94, 'model_version': 'hybrid-eta-v1'}


In [ ]:
test_requests = [
    {
        "train_id": "12345",
        "target_station_id": "STATION_A",
        "observed_at": "2026-09-04T12:00:00+05:30",
        "current_speed_kmph": 60,
        "average_speed_last_5_minutes": 58,
        "current_delay_minutes": 5,
        "distance_to_target_km": 80,
        "scheduled_time_to_target_minutes": 90,
        "rainfall_mm": 0,
        "congestion_score": 0.2,
        "historical_section_average_minutes": 95,
    },
    {
        "train_id": "12345",
        "target_station_id": "STATION_B",
        "observed_at": "2026-09-04T13:00:00+05:30",
        "current_speed_kmph": 30,
        "average_speed_last_5_minutes": 35,
        "current_delay_minutes": 20,
        "distance_to_target_km": 50,
        "scheduled_time_to_target_minutes": 55,
        "rainfall_mm": 15,
        "congestion_score": 0.8,
        "historical_section_average_minutes": 60,
    },
]

results = [predict_eta(request) for request in test_requests]

pd.DataFrame(results)

,train_id,target_station_id,predicted_remaining_minutes,lower_bound_minutes,upper_bound_minutes,model_version
0,12345,STATION_A,95.11,78.64,111.57,hybrid-eta-v1
1,12345,STATION_B,100.96,78.64,123.28,hybrid-eta-v1


In [ ]:
# save pipeline configuration

PIPELINE_CONFIG = {
    "model_version": MODEL_VERSION,
    "description": "Hybrid ETA pipeline using live "
    "ETAPredictionRequest data and "
    "historical railway delay context.",
    "delay_risk_model": risk_metadata["model_version"],
    "required_request_fields": REQUIRED_FIELDS,
    "optional_request_fields": [
        "rainfall_mm",
        "congestion_score",
        "historical_section_average_minutes",
    ],
    "pipeline_components": [
        "scheduled_baseline",
        "historical_section_eta",
        "live_speed_eta",
        "current_delay_propagation",
        "rainfall_adjustment",
        "congestion_adjustment",
        "prediction_interval",
    ],
}


with open(MODEL_DIR / "eta_pipeline_config.json", "w") as f:

    json.dump(PIPELINE_CONFIG, f, indent=2)

In [ ]:
# What we have now

# After the first two notebooks:

# Historical Railway Dataset
#         │
#         ▼
# ┌───────────────────────────┐
# │ Historical Analysis       │
# │                           │
# │ - train delay patterns    │
# │ - route punctuality       │
# │ - operational factors     │
# │ - weather factors         │
# └─────────────┬─────────────┘
#               │
#               ▼
#      XGBoost Delay Model
#               │
#               ▼
#    Historical Delay Context
#               │
#               │
# Live Train System ─────────────────┐
#                                    │
#                                    ▼
#                          ETAPredictionRequest
#                                   │
#                                   ▼
#                          Hybrid ETA Pipeline
#                                   │
#                                   ▼
#                          ETAPredictionResult


# Important limitation
# This hybrid-eta-v1 is a valid prototype aligned with your schemas,
# but the final production model should eventually replace these manually chosen coefficients:

# 0.40 scheduled ETA
# 0.25 historical ETA
# 0.35 live speed ETA